<a href="https://colab.research.google.com/github/MPfornow/RL/blob/main/notebooks/unit7/unit7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Code & Explanation:
https://huggingface.co/learn/deep-rl-course/en/unit7/hands-on

Env (SoccerTwos):
https://github.com/Unity-Technologies/ml-agents/blob/develop/docs/Learning-Environment-Examples.md#soccer-twos

Goal:
get the ball into the opponent’s goal while preventing the ball from entering your own goal.

**Note on Running Unit 7**

The SoccerTwos environment requires a relatively long training time and the available Google Colab runtime is insufficient for the full training process.

Therefore, the training section of this notebook is intended to be run locally using the provided ML-Agents environment and configuration. The commands below are kept unchanged from the local setup.

The notebook is provided primarily to document the implementation, configuration, and training procedure.

# Installations
git clone https://github.com/Unity-Technologies/ml-agents

cmd:
- winget install --id=astral-sh.uv -e
- inside "ml-agents" dictionary: ((uv venv .venv --python 3.10.12))
- .venv\Scripts\python.exe -m ensurepip --upgrade
- .venv\Scripts\python.exe -m pip install -e .\ml-agents-envs
- .venv\Scripts\python.exe -m pip install -e .\ml-agents

install git-lfs: https://git-lfs.com/

your executable should be in ml-agents/training-envs-executables/SoccerTwos:
https://drive.google.com/file/d/1sqFxbEdTMubjVktnV4C6ICjp89wLhUcP/view?usp=sharing

In [ ]:
!git clone https://github.com/Unity-Technologies/ml-agents

Cloning into 'ml-agents'...
Updating files:  57% (1277/2219)
Updating files:  58% (1288/2219)
Updating files:  59% (1310/2219)
Updating files:  60% (1332/2219)
Updating files:  61% (1354/2219)
Updating files:  62% (1376/2219)
Updating files:  63% (1398/2219)
Updating files:  64% (1421/2219)
Updating files:  65% (1443/2219)
Updating files:  66% (1465/2219)
Updating files:  67% (1487/2219)
Updating files:  68% (1509/2219)
Updating files:  69% (1532/2219)
Updating files:  70% (1554/2219)
Updating files:  71% (1576/2219)
Updating files:  72% (1598/2219)
Updating files:  73% (1620/2219)
Updating files:  74% (1643/2219)
Updating files:  75% (1665/2219)
Updating files:  76% (1687/2219)
Updating files:  77% (1709/2219)
Updating files:  78% (1731/2219)
Updating files:  79% (1754/2219)
Updating files:  80% (1776/2219)
Updating files:  81% (1798/2219)
Updating files:  82% (1820/2219)
Updating files:  83% (1842/2219)
Updating files:  84% (1864/2219)
Updating files:  85% (1887/2219)
Updating files:

In [ ]:
!ml-agents\.venv\Scripts\python.exe -m pip install torch~=2.2.1 --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121
     ---------------------------------------- 0.0/2.5 GB ? eta -:--:--
     ---------------------------------------- 0.0/2.5 GB 1.3 MB/s eta 0:30:58
     ---------------------------------------- 0.0/2.5 GB 1.3 MB/s eta 0:30:58
     ---------------------------------------- 0.0/2.5 GB 1.3 MB/s eta 0:30:58
     ---------------------------------------- 0.0/2.5 GB 581.0 kB/s eta 1:10:26
     ---------------------------------------- 0.0/2.5 GB 654.9 kB/s eta 1:02:29
     ---------------------------------------- 0.0/2.5 GB 654.9 kB/s eta 1:02:29
     ---------------------------------------- 0.0/2.5 GB 654.9 kB/s eta 1:02:29
     ---------------------------------------- 0.0/2.5 GB 425.3 kB/s eta 1:36:12
     ---------------------------------------- 0.0/2.5 GB 535.8 kB/s eta 1:16:22
     ---------------------------------------- 0.0/2.5 GB 628.5 kB/s eta 1:05:06
     ---------------------------------------- 0.0/2.5 GB 628.5 kB/s eta 1:05

# Config (Hyperparameters)
https://github.com/Unity-Technologies/ml-agents/blob/release_20_docs/docs/Training-Configuration-File.md

The config file we’re going to use here is in ./config/poca/SoccerTwos.yaml. It looks like this:

behaviors:
  SoccerTwos:
    trainer_type: poca
    hyperparameters:
      batch_size: 2048
      buffer_size: 20480
      learning_rate: 0.0003
      beta: 0.005
      epsilon: 0.2
      lambd: 0.95
      num_epoch: 3
      learning_rate_schedule: constant
    network_settings:
      normalize: false
      hidden_units: 512
      num_layers: 2
      vis_encode_type: simple
    reward_signals:
      extrinsic:
        gamma: 0.99
        strength: 1.0
    keep_checkpoints: 5
    max_steps: 5000000
    time_horizon: 1000
    summary_freq: 10000
    self_play:
      save_steps: 50000
      team_change: 200000
      swap_steps: 2000
      window: 10
      play_against_latest_model_ratio: 0.5
      initial_elo: 1200.0

In [ ]:
!ml-agents\.venv\Scripts\python.exe -c "import torch; print(torch.cuda.is_available()); print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')"

Train

In [ ]:
# !mlagents-learn "./config/poca/SoccerTwos.yaml" --env="./training-envs-executables/SoccerTwos.exe" --run-id="SoccerTwos" --no-graphics
!ml-agents/.venv/Scripts/mlagents-learn.exe "ml-agents/config/poca/SoccerTwos.yaml" --env="ml-agents/training-envs-executables/SoccerTwos.exe" --run-id="SoccerTwos" --no-graphics --torch-device=cuda

# Push

In [ ]:
# !huggingface-cli login
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# mlagents-push-to-hf  --run-id="SoccerTwos" --local-dir="./results/SoccerTwos" --repo-id="MP4good/poca-SoccerTwos" --commit-message="certification push"`
!ml-agents/.venv/Scripts/mlagents-push-to-hf.exe --run-id="SoccerTwos" --local-dir="ml-agents/results/SoccerTwos" --repo-id="MP4good/poca-SoccerTwos" --commit-message="certification push"

AI vs AI Challenge:

- have this tag in your model: ML-Agents-SoccerTwos, If not available, modify readme and add ((ML-Agents-SoccerTwos)) under "tags:"

- have a SoccerTwos.onnx file in "Files and versions" tab

- demo: https://huggingface.co/spaces/unity/ML-Agents-SoccerTwos